<a href="https://colab.research.google.com/github/Rudra-rudie/RNN-01/blob/main/NextWordPred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
pip install nltk


In [52]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [53]:
pip install airllm

In [54]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [55]:
with open("LICENSE.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace("document = ", "", 1)  # remove that first prefix

In [56]:
text

'Creative Commons Legal Code\n\nAttribution-ShareAlike 3.0 Unported\n\n    CREATIVE COMMONS CORPORATION IS NOT A LAW FIRM AND DOES NOT PROVIDE\n    LEGAL SERVICES. DISTRIBUTION OF THIS LICENSE DOES NOT CREATE AN\n    ATTORNEY-CLIENT RELATIONSHIP. CREATIVE COMMONS PROVIDES THIS\n    INFORMATION ON AN "AS-IS" BASIS. CREATIVE COMMONS MAKES NO WARRANTIES\n    REGARDING THE INFORMATION PROVIDED, AND DISCLAIMS LIABILITY FOR\n    DAMAGES RESULTING FROM ITS USE.\n\nLicense\n\nTHE WORK (AS DEFINED BELOW) IS PROVIDED UNDER THE TERMS OF THIS CREATIVE\nCOMMONS PUBLIC LICENSE ("CCPL" OR "LICENSE"). THE WORK IS PROTECTED BY\nCOPYRIGHT AND/OR OTHER APPLICABLE LAW. ANY USE OF THE WORK OTHER THAN AS\nAUTHORIZED UNDER THIS LICENSE OR COPYRIGHT LAW IS PROHIBITED.\n\nBY EXERCISING ANY RIGHTS TO THE WORK PROVIDED HERE, YOU ACCEPT AND AGREE\nTO BE BOUND BY THE TERMS OF THIS LICENSE. TO THE EXTENT THIS LICENSE MAY\nBE CONSIDERED TO BE A CONTRACT, THE LICENSOR GRANTS YOU THE RIGHTS\nCONTAINED HERE IN CONSIDER

In [57]:
#Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [58]:
#Tokenize
tokens =word_tokenize(text.lower())

In [59]:
 vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab   # no space

{'<unk>': 0,
 'creative': 1,
 'commons': 2,
 'legal': 3,
 'code': 4,
 'attribution-sharealike': 5,
 '3.0': 6,
 'unported': 7,
 'corporation': 8,
 'is': 9,
 'not': 10,
 'a': 11,
 'law': 12,
 'firm': 13,
 'and': 14,
 'does': 15,
 'provide': 16,
 'services': 17,
 '.': 18,
 'distribution': 19,
 'of': 20,
 'this': 21,
 'license': 22,
 'create': 23,
 'an': 24,
 'attorney-client': 25,
 'relationship': 26,
 'provides': 27,
 'information': 28,
 'on': 29,
 '``': 30,
 'as-is': 31,
 "''": 32,
 'basis': 33,
 'makes': 34,
 'no': 35,
 'warranties': 36,
 'regarding': 37,
 'the': 38,
 'provided': 39,
 ',': 40,
 'disclaims': 41,
 'liability': 42,
 'for': 43,
 'damages': 44,
 'resulting': 45,
 'from': 46,
 'its': 47,
 'use': 48,
 'work': 49,
 '(': 50,
 'as': 51,
 'defined': 52,
 'below': 53,
 ')': 54,
 'under': 55,
 'terms': 56,
 'public': 57,
 'ccpl': 58,
 'or': 59,
 'protected': 60,
 'by': 61,
 'copyright': 62,
 'and/or': 63,
 'other': 64,
 'applicable': 65,
 'any': 66,
 'than': 67,
 'authorized': 68,


In [60]:
len(vocab)

709

In [61]:
 input_sentences = []
 for sentence in text.split('\n'):
  input_sentences

In [62]:
input_sentences = text.split('\n')

In [63]:
def text_to_indices(sentence, vocab):
  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence



In [64]:
input_numerical_sentences = []
for sentence in input_sentences:
   input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

In [65]:
len(input_numerical_sentences)

362

In [66]:
training_sequence = []
for sentence in input_numerical_sentences:
    for i in range(1, len(sentence)):
        training_sequence.append(sentence[:i+1])
        print(sentence[:i+1])  # optional, just to visualize

len(training_sequence)

[1, 2]
[1, 2, 3]
[1, 2, 3, 4]
[5, 6]
[5, 6, 7]
[1, 2]
[1, 2, 8]
[1, 2, 8, 9]
[1, 2, 8, 9, 10]
[1, 2, 8, 9, 10, 11]
[1, 2, 8, 9, 10, 11, 12]
[1, 2, 8, 9, 10, 11, 12, 13]
[1, 2, 8, 9, 10, 11, 12, 13, 14]
[1, 2, 8, 9, 10, 11, 12, 13, 14, 15]
[1, 2, 8, 9, 10, 11, 12, 13, 14, 15, 10]
[1, 2, 8, 9, 10, 11, 12, 13, 14, 15, 10, 16]
[3, 17]
[3, 17, 18]
[3, 17, 18, 19]
[3, 17, 18, 19, 20]
[3, 17, 18, 19, 20, 21]
[3, 17, 18, 19, 20, 21, 22]
[3, 17, 18, 19, 20, 21, 22, 15]
[3, 17, 18, 19, 20, 21, 22, 15, 10]
[3, 17, 18, 19, 20, 21, 22, 15, 10, 23]
[3, 17, 18, 19, 20, 21, 22, 15, 10, 23, 24]
[25, 26]
[25, 26, 18]
[25, 26, 18, 1]
[25, 26, 18, 1, 2]
[25, 26, 18, 1, 2, 27]
[25, 26, 18, 1, 2, 27, 21]
[28, 29]
[28, 29, 24]
[28, 29, 24, 30]
[28, 29, 24, 30, 31]
[28, 29, 24, 30, 31, 32]
[28, 29, 24, 30, 31, 32, 33]
[28, 29, 24, 30, 31, 32, 33, 18]
[28, 29, 24, 30, 31, 32, 33, 18, 1]
[28, 29, 24, 30, 31, 32, 33, 18, 1, 2]
[28, 29, 24, 30, 31, 32, 33, 18, 1, 2, 34]
[28, 29, 24, 30, 31, 32, 33, 18, 1, 2, 34, 

3580

In [67]:
len_list = []
for sequnece in training_sequence:
  len_list.append(len(sequnece))

max(len_list)

23

In [68]:
padded_training_sequence = []
for sequnece in training_sequence:
  padded_training_sequence.append([0]*(max(len_list)-len(sequnece))+sequnece)


In [69]:
padded_training_sequence=torch.tensor(padded_training_sequence,dtype=torch.long)

In [70]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0,   0,  ..., 177, 178, 179],
        [  0,   0,   0,  ..., 178, 179, 708],
        [  0,   0,   0,  ..., 179, 708,  18]])

In [71]:
x = padded_training_sequence[:,:-1]
y = padded_training_sequence[:,-1]

In [72]:
x

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   0,  ..., 707, 177, 178],
        [  0,   0,   0,  ..., 177, 178, 179],
        [  0,   0,   0,  ..., 178, 179, 708]])

In [73]:
y

tensor([  2,   3,   4,  ..., 179, 708,  18])

In [74]:
class CustomDataset(Dataset):

  def __init__(self,x,y):
    self.x = x
    self.y = y
  def __len__(self):
    return self.x.shape[0]

  def __getitem__(self, idx):
    return self.x[idx], self.y[idx]

In [75]:
dataset = CustomDataset(x,y)

In [76]:
len(dataset)

3580

In [77]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 tensor(2))

In [78]:
dataloader = DataLoader(dataset , batch_size=32, shuffle=True)

In [79]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x)
        return x

In [80]:
x = nn.Embedding(352,100)
y = nn.LSTM(100,100,batch_first=True)


In [81]:
a = dataset[0][0].unsqueeze(0)

In [82]:
b =x(a)

In [83]:
c,d = y(b)

In [84]:
c.shape

torch.Size([1, 22, 100])

In [85]:
e ,f = d


In [86]:
e.shape

torch.Size([1, 1, 100])

In [87]:
f.shape

torch.Size([1, 1, 100])

In [88]:
print(d)
print(len(d))

(tensor([[[ 0.1679, -0.2454, -0.0160,  0.1923, -0.0876, -0.2981, -0.1656,
          -0.1929, -0.0494, -0.1327,  0.0180,  0.0186,  0.1139, -0.1729,
           0.0983,  0.0905, -0.0716,  0.2032, -0.1366, -0.1013, -0.1086,
           0.2201, -0.0035,  0.1260,  0.1771, -0.0163, -0.2118, -0.0039,
          -0.1283, -0.1033,  0.0685, -0.0263, -0.2487, -0.0977, -0.0021,
          -0.0970,  0.1442, -0.0483,  0.0816, -0.1468,  0.2102, -0.0869,
           0.0337,  0.0632, -0.0938, -0.1115,  0.2944, -0.1787,  0.4361,
          -0.0324,  0.2039,  0.0969,  0.0650,  0.1936,  0.0571,  0.0927,
           0.2509, -0.2029,  0.1649, -0.1109,  0.0181, -0.1352,  0.1817,
          -0.0864,  0.0824,  0.0307,  0.0828,  0.0878,  0.0214, -0.1146,
           0.1984,  0.1167, -0.2553, -0.2298, -0.0011,  0.0464,  0.2651,
           0.0756, -0.0925, -0.2081, -0.2594,  0.1402, -0.1527,  0.0097,
          -0.0211,  0.0201,  0.2690,  0.2977, -0.2694,  0.1288, -0.0978,
           0.2973,  0.1817,  0.0207, -0.3583,  0.1

In [89]:
e, f = d


In [90]:
print(e.shape)
print(f.shape)

torch.Size([1, 1, 100])
torch.Size([1, 1, 100])


In [91]:
model = LSTMModel(vocab_size=len(vocab))

In [92]:
device =torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [93]:
# take any sample sequence from your dataset
sample = training_sequence[0]
input_tensor = torch.tensor(sample, dtype=torch.long).unsqueeze(0)  # shape (1, seq_len)
print(input_tensor.shape)

torch.Size([1, 2])


In [94]:
output, (e, f) = model.lstm(model.embedding(input_tensor))
out = model.fc(output[:, -1, :])
print(out.shape)

torch.Size([1, 709])


In [95]:
epochs = 50
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [96]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x[:, -1, :])  # ✅ take only last timestep → (batch, vocab_size)
        return x

In [97]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(vocab_size=len(vocab)).to(device)  # ✅ model on device

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch:{epoch+1}, Loss:{total_loss:.4f}")  # ✅ f-string

Epoch:1, Loss:613.8001
Epoch:2, Loss:521.9647
Epoch:3, Loss:475.0114
Epoch:4, Loss:433.1150
Epoch:5, Loss:396.4997
Epoch:6, Loss:363.2357
Epoch:7, Loss:332.2669
Epoch:8, Loss:303.4260
Epoch:9, Loss:276.2380
Epoch:10, Loss:250.4435
Epoch:11, Loss:226.7700
Epoch:12, Loss:204.4041
Epoch:13, Loss:184.4219
Epoch:14, Loss:165.2778
Epoch:15, Loss:148.0521
Epoch:16, Loss:132.8538
Epoch:17, Loss:118.7337
Epoch:18, Loss:106.2706
Epoch:19, Loss:94.7392
Epoch:20, Loss:84.7902
Epoch:21, Loss:76.1160
Epoch:22, Loss:68.3965
Epoch:23, Loss:61.4174
Epoch:24, Loss:54.9346
Epoch:25, Loss:49.9531
Epoch:26, Loss:45.4161
Epoch:27, Loss:41.2206
Epoch:28, Loss:37.6678
Epoch:29, Loss:34.5918
Epoch:30, Loss:32.0622
Epoch:31, Loss:29.4250
Epoch:32, Loss:27.6098
Epoch:33, Loss:25.8054
Epoch:34, Loss:24.2484
Epoch:35, Loss:22.7477
Epoch:36, Loss:21.5157
Epoch:37, Loss:20.5855
Epoch:38, Loss:19.7720
Epoch:39, Loss:18.6226
Epoch:40, Loss:17.7943
Epoch:41, Loss:17.2471
Epoch:42, Loss:16.6878
Epoch:43, Loss:16.1900
Ep

In [98]:
def prediction(model, vocab, text, seq_len=10, num_words=5):  # num_words = how many words to generate
    tokens = word_tokenize(text.lower())
    idx2word = {v: k for k, v in vocab.items()}

    for _ in range(num_words):
        # convert to indices
        numerical = [vocab.get(token, 0) for token in tokens]

        # padding / truncating
        if len(numerical) < seq_len:
            numerical = [0] * (seq_len - len(numerical)) + numerical
        else:
            numerical = numerical[-seq_len:]

        # predict
        input_tensor = torch.tensor(numerical, dtype=torch.long).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)

        predicted_idx = torch.argmax(output, dim=1).item()
        predicted_word = idx2word[predicted_idx]

        # ✅ append predicted word back to tokens
        tokens.append(predicted_word)

    # join and return full sentence
    full_sentence = " ".join(tokens)
    print(f"Generated sentence: {full_sentence}")
    return full_sentence

In [99]:
prediction(model, vocab,"DISTRIBUTION OF THIS LICENSE DOES NOT CREATE")

Generated sentence: distribution of this license does not create an adaptation , other than


'distribution of this license does not create an adaptation , other than'

In [100]:
import time
num_tokens = 10
input_text = "distribution of this license does not create an licenses basis action"

for i in range(num_tokens):
    output_text = prediction(model, vocab, input_text)
    print(output_text)
    input_text = output_text
    time.sleep(0.3)

Generated sentence: distribution of this license does not create an licenses basis action prejudicial to the extent out
distribution of this license does not create an licenses basis action prejudicial to the extent out
Generated sentence: distribution of this license does not create an licenses basis action prejudicial to the extent out of this license despite intended
distribution of this license does not create an licenses basis action prejudicial to the extent out of this license despite intended
Generated sentence: distribution of this license does not create an licenses basis action prejudicial to the extent out of this license despite intended to reduce , this license
distribution of this license does not create an licenses basis action prejudicial to the extent out of this license despite intended to reduce , this license
Generated sentence: distribution of this license does not create an licenses basis action prejudicial to the extent out of this license despite intended to re

In [101]:
def calculate_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            output = model(batch_x)  # (batch, vocab_size)

            predicted = torch.argmax(output, dim=1)  # get highest prob word index

            correct += (predicted == batch_y).sum().item()  # count correct predictions
            total += batch_y.size(0)

    accuracy = (correct / total) * 100
    print(f"Accuracy: {accuracy:.2f}%")
    return accuracy

In [102]:
calculate_accuracy(model, dataloader, device)

Accuracy: 96.51%


96.50837988826815